# PACE Phase 11E5 — AMPGAN v2 reconstructed inference

**Analysis only — no manuscript edits.** This notebook restores the **pinned official AMPGAN v2 (v2.0)** checkpoint and original conditioning data. It bypasses a historical TensorFlow SavedModel graph placeholder by running the original documented generator architecture directly on **SHA-256-verified original weight tensors**. It does not retrain any model.

**Important scientific caveat:** The reconstructed path is not established as fully equivalent to native old-TensorFlow inference; the notebook explicitly labels it as reconstructed. Source/model/shape validation failures produce diagnostics, not fabricated peptide FASTA.


In [ ]:
from pathlib import Path
ROOT = Path('/content')
ROOT.mkdir(exist_ok=True)
(ROOT / 'PACE_Phase11E4_Official_Preprocess_and_Inference.py').write_text('"""PACE Phase 11E4: AMPGAN v2, pinned original-data preprocessing and SavedModel inference.\n\nSource release: https://gitlab.com/vail-uvm/amp-gan.git tag v2.0\nPinned commit: 2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\nAnalysis only; NEVER modifies original PACE materials. Does not train the GAN.\n\nThis script uses original source code for conditioning and decoding. If that source\nfails under the current runtime, it writes a diagnostic ZIP instead of inventing\nmetadata or generated peptides.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, json, os, platform, random, subprocess, sys, traceback, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nROOT=Path(\'/content\') if Path(\'/content\').exists() else Path.cwd()\nREPO=ROOT/\'ampgan_v2_official\'\nOUT=ROOT/\'PACE_Phase11E4_Results\'\nOUT.mkdir(parents=True,exist_ok=True)\nOUTPUT_ZIP=ROOT/\'PACE_Phase11E4_Results.zip\'\nREMOTE=\'https://gitlab.com/vail-uvm/amp-gan.git\'\nTAG=\'v2.0\'\nCOMMIT=\'2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\'\nMODEL=\'models/amp_gan_2020-07-08_24/gan_1606\'\nEXPECTED_WEIGHTS={\n    MODEL+\'/saved_model.pb\':(\'a5e7cafa16c33010bd0fe22747c89f05d74f8781deac9eeeb91c6e4c371177e2\',1344320),\n    MODEL+\'/variables/variables.data-00000-of-00002\':(\'df6b16408b05f21a6fdb1d675c12562257d619382d08af2fad391d7ed847084a\',53347),\n    MODEL+\'/variables/variables.data-00001-of-00002\':(\'012fc17624f1fe6b0441622bebc71ee968e4d7db80bb7910e22af93b49d69890\',12982248),\n    MODEL+\'/variables/variables.index\':(\'7936298d0a7ac1802bbedc726d29e27911c3aa60778ac97395a24d5f82456902\',12646),\n}\nSOURCE_SHA={\n \'ampgan/dbaasp.py\':\'f8a2abefd20e04c07b4723b11dfa985bb9883ae56b99633f915805c9fe494e7d\',\n \'ampgan/avpdb.py\':\'c91c621c992d4a6aa414af06dd3187d0bf827de5be2dcabee38ca7ef5f0c45c0\',\n \'ampgan/uniprot.py\':\'adaf9dd191d020a67889a676b1db4d847daeb5c723fdc9f1d99d12c5f8f9a60c\',\n \'ampgan/data_utils.py\':\'f6e9cedf86016390c48eefc94b721349b9a39a694475784562fea4c5ec2af525\',\n \'ampgan/generate_samples.py\':\'4590d9cbfd42c5b0e64fb000caf7e80d6aada6aacd23d08f4fae8dd143a00c21\',\n}\nBATCH_SIZE=int(os.environ.get(\'PACE_11E4_BATCH\',\'128\'))\nTARGET_POS=int(os.environ.get(\'PACE_11E4_POSITIVE_COUNT\',\'1000\'))\nSEED=20261114\nstatus={\'phase\':\'11E4\',\'scope\':\'original-source-positive-conditioned AMPGAN v2 sampling\',\n    \'source\':REMOTE,\'tag\':TAG,\'pinned_commit\':COMMIT,\n    \'model_checkpoint\':MODEL,\'python\':sys.version.split()[0],\n    \'platform\':platform.platform(), \'seed\':SEED, \'batch_size\':BATCH_SIZE,\n    \'target_positive_conditioned\':TARGET_POS,\n    \'generated_count\':0,\'canonical_positive_count\':0,\'unique_canonical_positive_count\':0,\n    \'source_sha256\':{},\'data_sources\':{},\'checkpoint_sha256\':{},\'stages\':{},\n    \'notes\':[\'No PACE manuscript, supplementary data or reviewer-response edits\',\n             \'Sequences are generated hypotheses, not experimentally verified antimicrobial peptides\',\n             \'Uses positive/negative source conditions in original PairedAMPSequence order; comparison benchmark uses positive-conditioned subset only\',\n             \'Modern runtime may be incompatible with historical original preprocessing; any incompatibility is recorded, not silently patched\']}\nlogs=[]\n\ndef log(msg):\n    msg=str(msg);print(msg,flush=True);logs.append(msg)\n\ndef run(args,cwd=None,env=None,timeout=3600,allow_fail=False):\n    log(\'$ \'+\' \'.join(map(str,args)))\n    p=subprocess.run(args,cwd=str(cwd) if cwd is not None else None,\n                     env=env,text=True,stdout=subprocess.PIPE,\n                     stderr=subprocess.STDOUT,timeout=timeout)\n    log(\'exit=%s | output tail:\\n%s\' % (p.returncode,(p.stdout or \'\')[-5000:]))\n    if p.returncode and not allow_fail:\n        raise RuntimeError(\'Command failed: \'+\' \'.join(map(str,args)) + \'\\n\'+(p.stdout or \'\')[-1200:])\n    return p\n\ndef digest(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1<<20),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef pointer(path):\n    if not path.exists(): return True\n    with path.open(\'rb\') as f: return f.read(100).startswith(b\'version https://git-lfs.github.com/spec/v1\')\n\ndef setup():\n    if run([\'git\',\'lfs\',\'version\'],allow_fail=True).returncode:\n        log(\'Git LFS not present; installing...\')\n        run([\'apt-get\',\'update\',\'-qq\'],timeout=800,allow_fail=True)\n        run([\'apt-get\',\'install\',\'-y\',\'-qq\',\'git-lfs\'],timeout=1200)\n        run([\'git\',\'lfs\',\'version\'])\n    if not (REPO/\'.git\').is_dir():\n        e=os.environ.copy();e[\'GIT_LFS_SKIP_SMUDGE\']=\'1\'\n        run([\'git\',\'clone\',\'--branch\',TAG,\'--depth\',\'1\',REMOTE,str(REPO)],env=e,timeout=1600)\n    commit=run([\'git\',\'rev-parse\',\'HEAD\'],cwd=REPO).stdout.strip()\n    if commit!=COMMIT: raise RuntimeError(\'Unexpected Git commit \'+commit+\'; refusing to continue\')\n    status[\'actual_commit\']=commit\n    run([\'git\',\'lfs\',\'install\',\'--local\'],cwd=REPO)\n    for rel,expected in SOURCE_SHA.items():\n        p=REPO/rel\n        if not p.exists() or digest(p)!=expected:raise RuntimeError(\'Official source SHA256 mismatch: \'+rel)\n        status[\'source_sha256\'][rel]=expected\n    status[\'stages\'][\'source\']=\'verified\'\n\n    need=[rel for rel,(h,n) in EXPECTED_WEIGHTS.items()\n          if not (REPO/rel).exists() or (REPO/rel).stat().st_size!=n or digest(REPO/rel)!=h]\n    if need:\n        run([\'git\',\'lfs\',\'pull\',\'--include\',MODEL+\'/**\',\'--exclude\',\'\'],cwd=REPO,timeout=3600)\n    for rel,(h,n) in EXPECTED_WEIGHTS.items():\n        p=REPO/rel\n        if not p.exists() or p.stat().st_size!=n or digest(p)!=h:\n            raise RuntimeError(\'Official model checkpoint verification failed: \'+rel)\n        status[\'checkpoint_sha256\'][rel]=h\n    status[\'stages\'][\'checkpoint\']=\'verified\'\n\n    # Official input files, NOT processed data. LFS pointers are not acceptable.\n    raw={\n      \'data/dbaasp/raw.json\':None,\n      \'data/avpdb/AVPdb_data.tsv\':731264,\n      \'data/avpdb/mAVPdb_data.tsv\':117772,\n      \'data/uniprot/uniprot.tab\':47128557,\n    }\n    lfs=[k for k in raw if pointer(REPO/k)]\n    if lfs:\n        run([\'git\',\'lfs\',\'pull\',\'--include\',\'data/dbaasp/**,data/avpdb/**,data/uniprot/**\',\'--exclude\',\'\'],cwd=REPO,timeout=3600)\n    for rel,size in raw.items():\n        p=REPO/rel\n        if not p.exists() or pointer(p):\n            raise FileNotFoundError(\'Official reference data not retrieved: \'+rel)\n        if size is not None and p.stat().st_size!=size:\n            raise RuntimeError(\'Unexpected raw LFS dataset size for \'+rel+\': \'+str(p.stat().st_size))\n        source_hash=digest(p)\n        # Independent Git-LFS object verification against the PINNED Git tree.\n        git_blob=run([\'git\',\'show\',\'HEAD:\'+rel],cwd=REPO).stdout\n        lfs_oid=None\n        if git_blob.startswith(\'version https://git-lfs.github.com/spec/v1\'):\n            for line in git_blob.splitlines():\n                if line.startswith(\'oid sha256:\'):lfs_oid=line.partition(\':\')[2].strip()\n            if not lfs_oid or source_hash!=lfs_oid:\n                raise RuntimeError(\'Git LFS OID mismatch for source dataset \'+rel)\n        status[\'data_sources\'][rel]={\'bytes\':p.stat().st_size,\'sha256\':source_hash,\n                                     \'pinned_git_lfs_sha256\':lfs_oid}\n    status[\'stages\'][\'reference_data\']=\'verified\'\n\ndef preprocess():\n    """Only recreate upstream DBAASP cleaned table; AVPdb/UniProt are loaded as TSVs."""\n    import numpy as np\n    import pandas as pd\n    source=REPO/\'ampgan\'\n    sys.path.insert(0,str(source))\n    os.chdir(source) # upstream uses hardcoded ../ relative paths\n    import dbaasp, avpdb, uniprot, data_utils\n    status[\'runtime\']={\'numpy\':np.__version__,\'pandas\':pd.__version__}\n    # The original source expects these tracked target maps for decoding labels.\n    for f in [REPO/\'data/dbaasp/targets_mapping.json\', REPO/\'data/avpdb/targets_mapping.json\']:\n        if not f.exists() or pointer(f):\n            raise FileNotFoundError(\'Required original upstream target map missing: \'+str(f))\n    data_clean=REPO/\'data/dbaasp/clean.csv\'\n    # Preprocess only when needed. This is the exact original function.\n    if not data_clean.exists() or data_clean.stat().st_size<50:\n        log(\'Generating missing DBAASP clean.csv using original dbaasp.prepare()...\')\n        dbaasp.prepare()\n    if not data_clean.exists() or data_clean.stat().st_size<50:\n        raise RuntimeError(\'Original dbaasp.prepare() failed to produce meaningful clean.csv\')\n    status[\'dbassp_clean\']={\'size\':data_clean.stat().st_size,\'sha256\':digest(data_clean)}\n    # Validations call ORIGINAL load_data functions, no fabricated conditions.\n    df_d=dbaasp.load_data()\n    df_a=avpdb.load_data()\n    df_u=uniprot.load_data(amp_sequences=pd.concat([df_d.sequence,df_a.sequence]))\n    status[\'curation_counts\']={\'DBAASP\':len(df_d),\'AVPdb\':len(df_a),\'UniProt\':len(df_u)}\n    if min(len(df_d),len(df_a),len(df_u))<1:raise RuntimeError(\'One upstream dataset is empty\')\n    log(\'Original data loaded: DBAASP=%d AVPdb=%d UniProt=%d\'%(len(df_d),len(df_a),len(df_u)))\n    data=data_utils.get_train_data(BATCH_SIZE)\n    if len(data)<1:raise RuntimeError(\'Original PairedAMPSequence is empty\')\n    _,cond=data[0]\n    cond=np.asarray(cond)\n    if cond.ndim!=2 or len(cond)!=BATCH_SIZE:raise RuntimeError(\'Unexpected official condition matrix shape \'+str(cond.shape))\n    status[\'conditioning\']={\'batch_shape\':list(cond.shape),\'batches\':len(data),\n            \'positive_first_half\':True,\'source\':\'original data_utils.get_train_data\'}\n    for rel in (\'results/targets_values.json\',\'results/target_groups_values.json\',\'results/mic50_bins.json\'):\n        p=REPO/rel\n        if not p.is_file():raise FileNotFoundError(\'Original conditioning metadata not created: \'+rel)\n        json.loads(p.read_text())\n        status[\'conditioning\'][rel]={\'sha256\':digest(p),\'bytes\':p.stat().st_size}\n    status[\'stages\'][\'original_preprocessing\']=\'passed\'\n    return data_utils,data\n\ndef generate(du,data):\n    import numpy as np\n    import tensorflow as tf\n    status[\'runtime\'][\'tensorflow\']=tf.__version__\n    random.seed(SEED);np.random.seed(SEED);tf.random.set_seed(SEED)\n    log(\'Loading official verified SavedModel via tf.saved_model.load...\')\n    root=tf.saved_model.load(str(REPO/MODEL))\n    gen=getattr(root,\'generator\',None)\n    if not callable(gen):raise RuntimeError(\'SavedModel generator not callable\')\n    status[\'stages\'][\'savedmodel\']=\'passed\'\n    canonical=set(\'ACDEFGHIKLMNPQRSTVWY\')\n    rows=[];positive=0\n    b=0\n    max_batches=max(2,(TARGET_POS+(BATCH_SIZE//2)-1)//(BATCH_SIZE//2)+2)\n    # Source\'s PairedAMPSequence has positive conditions first, negative second.\n    while positive<TARGET_POS and b<max_batches:\n        if b and b%len(data)==0:data.on_epoch_end()\n        _, conditions=data[b%len(data)]\n        conditions=np.asarray(conditions,dtype=\'float32\')\n        if len(conditions)!=BATCH_SIZE:raise RuntimeError(\'Unexpected batch size\')\n        latent=np.random.normal(size=(len(conditions),256)).astype(\'float32\')\n        generated=gen([tf.convert_to_tensor(latent),tf.convert_to_tensor(conditions)])\n        if isinstance(generated,dict):\n            if len(generated)!=1:raise RuntimeError(\'Generator has ambiguous multi-output dictionary\')\n            generated=next(iter(generated.values()))\n        if hasattr(generated,\'numpy\'):generated=generated.numpy()\n        generated=np.asarray(generated)\n        expected_chars=len(du.non_wildcard_symbols)+1\n        if generated.ndim!=3 or generated.shape[0]!=len(conditions) or generated.shape[-1]!=expected_chars:\n            raise RuntimeError(\'Unexpected generator output %s; expected [batch, length, %s]\'%(generated.shape,expected_chars))\n        sequences=du.decode_sequences(generated,concatenate=False)\n        half=len(conditions)//2\n        for i,(seq,cond) in enumerate(zip(sequences,conditions)):\n            ampcondition=(i<half)\n            if ampcondition and positive>=TARGET_POS:continue\n            if ampcondition:positive+=1\n            s=seq.strip()\n            rows.append({\'id\':\'AMPGANv2_%06d\'%(len(rows)+1),\n                         \'sequence\':s,\'length\':len(s),\n                         \'condition_source\':\'DBAASP_AVPdb_AMP\' if ampcondition else \'UniProt_nonAMP\',\n                         \'canonical\':int(bool(s) and set(s)<=canonical),\n                         \'condition_sha256\':hashlib.sha256(cond.tobytes()).hexdigest()})\n        log(\'Completed batch %d: positive-conditioned=%d, total=%d\'%(b+1,positive,len(rows)))\n        b+=1\n    if positive<TARGET_POS:raise RuntimeError(\'Could not reach requested target count\')\n    fields=[\'id\',\'sequence\',\'length\',\'condition_source\',\'canonical\',\'condition_sha256\']\n    with (OUT/\'ampganv2_generated_all_conditions.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)\n    positive_valid=[r for r in rows if r[\'condition_source\']==\'DBAASP_AVPdb_AMP\' and r[\'canonical\']]\n    with (OUT/\'ampganv2_positive_conditioned_canonical.fasta\').open(\'w\') as f:\n        for r in positive_valid:f.write(\'>\'+r[\'id\']+\'\\n\'+r[\'sequence\']+\'\\n\')\n    with (OUT/\'ampganv2_positive_conditioned_raw.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows([r for r in rows if r[\'condition_source\']==\'DBAASP_AVPdb_AMP\'])\n    status[\'generated_count\']=len(rows)\n    status[\'positive_conditioned_generated_count\']=positive\n    status[\'canonical_positive_count\']=len(positive_valid)\n    status[\'unique_canonical_positive_count\']=len({r[\'sequence\'] for r in positive_valid})\n    status[\'canonical_all_conditions_count\']=sum(r[\'canonical\'] for r in rows)\n    status[\'generator_output_shape\']=list(generated.shape)\n    status[\'stages\'][\'generation\']=\'passed\'\n    if not positive_valid:\n        raise RuntimeError(\'Generator returned no nonempty canonical AMP-conditioned sequences; see raw CSV\')\n    status[\'status\']=\'authentic_positive_conditioned_candidates_generated\'\n\ndef finalize():\n    status[\'timestamp_finished_utc\']=datetime.now(timezone.utc).isoformat()\n    (OUT/\'status.json\').write_text(json.dumps(status,indent=2,default=str))\n    (OUT/\'run.log\').write_text(\'\\n\'.join(logs)+\'\\n\')\n    (OUT/\'README.txt\').write_text(\n        \'PACE Phase 11E4 — original AMPGANv2 v2.0 data-conditioning and checkpoint inference.\\n\'\n        \'The original training corpus and model weights are NOT in this archive.\\n\'\n        \'Peptides are computational samples; no experimental antimicrobial or hemolysis claims.\\n\'\n        \'If status is blocked_with_diagnostics, NO candidate benchmark should be reported.\\n\'\n        \'Original PACE manuscript, supplements and reviewer responses are unmodified.\\n\')\n    with zipfile.ZipFile(OUTPUT_ZIP,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(OUT.iterdir()):\n            if path.is_file():z.write(path,arcname=path.name)\n    with zipfile.ZipFile(OUTPUT_ZIP) as z:assert z.testzip() is None\n    log(\'OUTPUT ZIP: %s (%d bytes)\'%(OUTPUT_ZIP,OUTPUT_ZIP.stat().st_size))\n\nif __name__==\'__main__\':\n    try:\n        if BATCH_SIZE%2:raise ValueError(\'batch must be even to match original PairedAMPSequence\')\n        setup()\n        du,data=preprocess()\n        generate(du,data)\n    except BaseException as err:\n        status[\'status\']=\'blocked_with_diagnostics\'\n        status[\'error_class\']=type(err).__name__\n        status[\'error_message\']=str(err)\n        (OUT/\'error_traceback.txt\').write_text(traceback.format_exc())\n        log(\'BLOCKED: \'+repr(err)+\'\\n\'+traceback.format_exc()[-4500:])\n    finally:\n        finalize()\n', encoding='utf-8')
(ROOT / 'PACE_Phase11E5_Verified_Architecture_Forward.py').write_text('"""AMPGAN v2 Phase 11E5 — verified weights and structural forward-pass recovery.\n\nReconstructs the ORIGINAL generator forward graph from original ampgan/model.py\nand uses ONLY variables from the PINNED AND SHA-256 VERIFIED SavedModel, as an\nalternative to broken obsolete coordinate_channel1d SavedModel concrete trace.\n\nThis is not the original executable SavedModel graph; outputs must be labeled\n\'manually reconstructed inference from official weights\'. No biological claims.\nNeither source PACE files nor source AMPGAN checkpoint are modified.\n"""\nfrom __future__ import annotations\nimport csv, hashlib, importlib.util, json, os, platform, re, sys, traceback, zipfile\nfrom collections import Counter\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nROOT=Path(\'/content\') if Path(\'/content\').exists() else Path.cwd()\nHERE=Path(__file__).parent\nOUT=ROOT/\'PACE_Phase11E5_Results\'\nOUT.mkdir(exist_ok=True)\n# Never retain stale candidate exports from a previously failed run.\nfor stale in OUT.glob(\'ampganv2_*\'):\n    if stale.is_file(): stale.unlink()\nE4_SCRIPT=HERE/\'PACE_Phase11E4_Official_Preprocess_and_Inference.py\'\nZIP=ROOT/\'PACE_Phase11E5_Results.zip\'\nSEED=20261114\nTARGET_POSITIVE=1000\nBATCH_SIZE=128\nstatus={\'phase\':\'11E5\',\'status\':\'initializing\',\'seed\':SEED,\'target_positive\':TARGET_POSITIVE,\n        \'batch_size\':BATCH_SIZE,\'execution_mode\':\'original_architecture_reconstructed_from_original_verified_weights\',\n        \'no_original_PACE_files_modified\':True,\'generated_count\':0,\'positive_canonical_count\':0,\n        \'positive_unique_canonical_count\':0,\'stages\':{},\'checks\':{},\'references\':{\n        \'AMPGANv2\':\'https://gitlab.com/vail-uvm/amp-gan.git; v2.0; 2102eb5d26dd9a3c12a5a0bad7c7bb3c58b80c70\',\n        \'coordinate_channel_1d\':\'https://github.com/titu1994/keras-coordconv/blob/master/coord.py\'}}\nlogs=[]\ndef log(s):\n    s=str(s);print(s,flush=True);logs.append(s)\n\ndef get_e4():\n    if not E4_SCRIPT.exists():raise FileNotFoundError(\'Original Phase11E4 runner missing: \'+str(E4_SCRIPT))\n    spec=importlib.util.spec_from_file_location(\'original_11e4_audit\',str(E4_SCRIPT))\n    e4=importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(e4)\n    return e4\n\ndef audit_reconstruction_sources(e4):\n    # Strict script/source pinning already happens in e4.setup, but here assert\n    # critical generator operations independently from official model.py.\n    official=(e4.REPO/\'ampgan/model.py\').read_text()\n    anchored=[\n      \'def amp_generator(\',\n      \'features = vailtools.layers.CoordinateChannel1D()(features)\',\n      \'layers.Dense(np.product(output_shape))\',\n      \'features = layers.Reshape(output_shape)(features)\',\n      \'kernel_size=3, dilation_rate=2 ** i, padding="same"\',\n      \'total_features.append(features)\',\n      \'layers.Concatenate()(total_features)\',\n      \'samples = layers.Activation(final_activation)(features)\',\n    ]\n    absent=[s for s in anchored if s not in official]\n    if absent:raise RuntimeError(\'Original generator architecture changed/unexpected; missing markers: \'+str(absent))\n    start=official.index(\'def amp_generator(\')\n    end=official.index(\'\\ndef amp_discriminator(\',start)\n    extracted=official[start:end]\n    # No claims about other source changes. Exact generator source hashed.\n    status[\'checks\'][\'generator_source_sha256\']=hashlib.sha256(extracted.encode()).hexdigest()\n    status[\'checks\'][\'original_generator_defaults\']={\n        \'output_shape\':[32,23],\'latent_shape\':[256],\'condition_shape\':[64],\n        \'blocks\':6,\'filters\':128,\'activation\':\'tanh\'}\n    log(\'Official amp_generator structural markers verified.\')\n\nEXPECTED=[(320,736),(736,), (3,24,128),(128,)] + sum(([ (3,128,128), (128,) ] for _ in range(5)),[]) + [(1,768,23),(23,)]\nassert len(EXPECTED)==16\n\ndef parse_conv_order(name):\n    # Names like conv1d/kernel, conv1d_1/kernel, generator/conv1d_4/bias.\n    m=re.search(r\'(?:^|/)conv1d(?:_(\\d+))?/(?:kernel|bias)(?::\\d+)?$\',name)\n    if not m:return None\n    return int(m.group(1) or 0)\n\ndef get_generator_weights(root,model_path):\n    import numpy as np\n    gen=getattr(root,\'generator\',None)\n    if gen is None:raise RuntimeError(\'SavedModel missing generator\')\n    vars_=list(getattr(gen,\'variables\',[]) or [])\n    log(\'SavedModel restored generator variable count: \'+str(len(vars_)))\n    rows=[]\n    for i,v in enumerate(vars_):\n        sh=tuple(int(s) for s in v.shape)\n        name=str(getattr(v,\'name\',\'\'))\n        rows.append({\'index\':i,\'name\':name,\'shape\':list(sh)})\n        log(\' var #%02d %-64s %s\'%(i,name,str(sh)))\n    status[\'variables\']=rows\n    (OUT/\'generator_restored_variable_inventory.json\').write_text(json.dumps(rows,indent=2))\n    if not vars_:\n        # Emit checkpoint entries for manual debugging instead of guessing mappings.\n        import tensorflow as tf\n        ckpt=str(model_path/\'variables/variables\')\n        names=tf.train.list_variables(ckpt)\n        (OUT/\'checkpoint_variable_inventory.json\').write_text(json.dumps(names,indent=2))\n        raise RuntimeError(\'Generator variables not exposed. Check checkpoint_variable_inventory.json\')\n\n    shapes=[tuple(int(s) for s in v.shape) for v in vars_]\n    if shapes==EXPECTED:\n        status[\'checks\'][\'variable_binding\']=\'native_generator_variables_in_exact_keras_layer_order\'\n        ordered=vars_\n    else:\n        # Name-based reconstruction: all conv names must be disambiguated.\n        byname={}\n        for v in vars_:\n            name=str(getattr(v,\'name\',\'\'))\n            m=re.search(r\'(?:^|/)(dense|conv1d(?:_\\d+)?)/(kernel|bias)(?::\\d+)?$\',name)\n            if m:\n                lay,role=m.groups()\n                key=(lay,role)\n                if key in byname:raise RuntimeError(\'Ambiguous duplicate generator variable \'+str(key))\n                byname[key]=v\n        expectedkeys=[(\'dense\',\'kernel\'),(\'dense\',\'bias\')]\n        for i in range(7):\n            lay=\'conv1d\'+(\'_\'+str(i) if i else \'\')\n            expectedkeys.extend([(lay,\'kernel\'),(lay,\'bias\')])\n        if len(byname)>=16 and all(k in byname for k in expectedkeys):\n            ordered=[byname[k] for k in expectedkeys]\n            if [tuple(int(s) for s in v.shape) for v in ordered]!=EXPECTED:\n                raise RuntimeError(\'Variable names resolve but weight tensor shapes mismatch\')\n            status[\'checks\'][\'variable_binding\']=\'unique_layer_names_and_expected_shapes\'\n        else:\n            raise RuntimeError(\'Generator variable ordering/identity cannot be verified; refusing guessed weight mapping. Shape sequence: \'+str(shapes))\n    weights=[tf_to_array(x) for x in ordered]\n    if [tuple(w.shape) for w in weights]!=EXPECTED:raise RuntimeError(\'Strict architecture weight-shape verification failed\')\n    if not all(np.isfinite(w).all() for w in weights):raise RuntimeError(\'Nonfinite generator weights\')\n    status[\'checks\'][\'all_16_weight_tensor_shapes_match_original_architecture\']=True\n    return weights\n\ndef tf_to_array(v):\n    import numpy as np\n    return np.asarray(v.numpy(),dtype=np.float32)\n\ndef forward(latent,conditions,weights):\n    import numpy as np\n    import tensorflow as tf\n    assert latent.ndim==2 and latent.shape[1]==256\n    assert conditions.ndim==2 and conditions.shape==(len(latent),64)\n    w=[tf.convert_to_tensor(t,dtype=tf.float32) for t in weights]\n    x=tf.concat([tf.convert_to_tensor(latent,dtype=tf.float32),tf.convert_to_tensor(conditions,dtype=tf.float32)],axis=-1)\n    x=tf.nn.leaky_relu(tf.linalg.matmul(x,w[0])+w[1],alpha=0.2)\n    x=tf.reshape(x,(-1,32,23))\n    # Exact rank-1 CoordConv algebra in the VailTools vendored upstream:\n    # range(0,32)/float32(31)*2-1; appended after the 23 original channels.\n    xx=tf.cast(tf.range(32),tf.float32)/tf.constant(31.0,tf.float32)\n    xx=(xx*2.0)-1.0\n    coord=tf.broadcast_to(tf.reshape(xx,(1,32,1)),(tf.shape(x)[0],32,1))\n    x=tf.concat([x,coord],axis=-1)\n    blocks=[]\n    for i in range(6):\n        k,b=w[2+2*i],w[3+2*i]\n        x=tf.nn.conv1d(x,k,stride=1,padding=\'SAME\',dilations=2**i)\n        x=tf.nn.leaky_relu(tf.nn.bias_add(x,b),alpha=0.2)\n        blocks.append(x)\n    x=tf.concat(blocks,axis=-1)\n    x=tf.nn.conv1d(x,w[14],stride=1,padding=\'SAME\')\n    x=tf.nn.tanh(tf.nn.bias_add(x,w[15]))\n    val=np.asarray(x.numpy())\n    if val.shape!=(len(latent),32,23) or not np.isfinite(val).all():\n        raise RuntimeError(\'Reconstructed generator invalid output: \'+str(val.shape))\n    if np.max(np.abs(val))>1.001:raise RuntimeError(\'Tanh boundedness violated\')\n    return val\n\ndef generate(e4,data,weights):\n    import numpy as np\n    import tensorflow as tf\n    np.random.seed(SEED);tf.random.set_seed(SEED)\n    du=sys.modules.get(\'data_utils\')\n    if du is None:raise RuntimeError(\'Original upstream data_utils module not imported\')\n    canonical=set(\'ACDEFGHIKLMNPQRSTVWY\')\n    rows=[];positive_count=0\n    max_batches=(TARGET_POSITIVE+(BATCH_SIZE//2)-1)//(BATCH_SIZE//2)+1\n    status[\'checks\'][\'non_wildcard_original_alphabet\']=\'\'.join(du.non_wildcard_symbols)\n    if len(du.non_wildcard_symbols)+1!=23:\n        raise RuntimeError(\'Official 23-channel decoding alphabet mismatch\')\n    for b in range(max_batches):\n        if positive_count>=TARGET_POSITIVE:break\n        if b and b%len(data)==0:data.on_epoch_end()\n        _,conditions=data[b%len(data)]\n        conditions=np.asarray(conditions,dtype=\'float32\')\n        if conditions.shape!=(BATCH_SIZE,64):raise RuntimeError(\'Unexpected original conditioning shape \'+str(conditions.shape))\n        z=np.random.normal(size=(BATCH_SIZE,256)).astype(\'float32\')\n        arr=forward(z,conditions,weights)\n        sequences=du.decode_sequences(arr,concatenate=False)\n        if len(sequences)!=BATCH_SIZE:raise RuntimeError(\'Original decoder gave incorrect sequence count\')\n        for j,(seq,cond) in enumerate(zip(sequences,conditions)):\n            is_positive=j<(BATCH_SIZE//2)\n            if is_positive and positive_count>=TARGET_POSITIVE:continue\n            if is_positive:positive_count+=1\n            seq=str(seq).strip()\n            rows.append({\'id\':\'AMPGANv2_recon_%06d\'%(len(rows)+1),\n             \'sequence\':seq,\'length\':len(seq),\'condition_type\':\'AMP_positive\' if is_positive else \'nonAMP_negative\',\n             \'canonical\':int(bool(seq) and set(seq)<=canonical),\n             \'condition_sha256\':hashlib.sha256(cond.tobytes()).hexdigest(),\n             \'inference_mode\':\'manual_verified_architecture_forward_from_official_weights\'})\n        log(\'Reconstructed inference batch %d; AMP positive=%d; total=%d\'%(b+1,positive_count,len(rows)))\n    if positive_count<TARGET_POSITIVE:raise RuntimeError(\'Did not generate required positive-conditioned count\')\n    fields=list(rows[0])\n    with (OUT/\'ampganv2_all_conditions_reconstructed.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows)\n    pos=[r for r in rows if r[\'condition_type\']==\'AMP_positive\']\n    with (OUT/\'ampganv2_positive_conditioned_raw.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(pos)\n    valid=[r for r in pos if r[\'canonical\']]\n    with (OUT/\'ampganv2_positive_conditioned_canonical.fasta\').open(\'w\') as f:\n        for r in valid:f.write(\'>\'+r[\'id\']+\'\\n\'+r[\'sequence\']+\'\\n\')\n    status[\'generated_count\']=len(rows)\n    status[\'positive_conditioned_generated_count\']=len(pos)\n    status[\'positive_canonical_count\']=len(valid)\n    status[\'positive_unique_canonical_count\']=len(set(r[\'sequence\'] for r in valid))\n    status[\'checks\'][\'positive_lengths\']=dict(Counter(r[\'length\'] for r in pos))\n    status[\'stages\'][\'reconstructed_generation\']=\'passed\'\n    status[\'status\']=\'reconstructed_candidates_available_pending_independent_validation\'\n    log(\'Reconstructed generation produced %d positive sequences, %d canonical, %d unique canonical\'%(\n        len(pos),len(valid),status[\'positive_unique_canonical_count\']))\n\ndef main():\n    try:\n        e4=get_e4()\n        e4.setup()\n        status[\'stages\'][\'source_reference_checkpoint\']=\'verified_via_original_phase11e4_sha256_checks\'\n        audit_reconstruction_sources(e4)\n        _,data=e4.preprocess()\n        status[\'stages\'][\'original_preprocessing\']=\'verified\'\n        import tensorflow as tf\n        status[\'runtime\']={\'tensorflow\':tf.__version__,\'python\':sys.version.split()[0]}\n        root=tf.saved_model.load(str(e4.REPO/e4.MODEL))\n        weights=get_generator_weights(root,e4.REPO/e4.MODEL)\n        log(\'All 16 generator variables validated; executing original architecture with direct TF ops.\')\n        generate(e4,data,weights)\n    except BaseException as exc:\n        status[\'status\']=\'blocked_with_diagnostics\'\n        status[\'error_type\']=type(exc).__name__\n        status[\'error_message\']=str(exc)\n        (OUT/\'error_traceback.txt\').write_text(traceback.format_exc())\n        log(\'BLOCKED: \'+type(exc).__name__+\': \'+str(exc))\n    finally:\n        status[\'finished_utc\']=datetime.now(timezone.utc).isoformat()\n        (OUT/\'status.json\').write_text(json.dumps(status,indent=2,default=str))\n        (OUT/\'run.log\').write_text(\'\\n\'.join(logs)+\'\\n\')\n        (OUT/\'README.txt\').write_text(\n            \'PACE Phase 11E5: manual graph reconstruction based on ORIGINAL AMPGAN v2 generator structure \'\n            \'and original verified checkpoint variables. This bypasses a broken historical CoordinateChannel1D \'\n            \'SavedModel graph placeholder. It does NOT validate equivalence to original TensorFlow 2.4 inference. \'\n            \'Do not describe these outputs as native official generator sampling without independent equivalence checks. \'\n            \'No experimental antimicrobial or hemolysis results. Original PACE files unchanged.\\n\')\n        with zipfile.ZipFile(ZIP,\'w\',zipfile.ZIP_DEFLATED) as zf:\n            for p in sorted(OUT.iterdir()):\n                if p.is_file():zf.write(p,arcname=p.name)\n        assert zipfile.ZipFile(ZIP).testzip() is None\n        log(\'Output ZIP: \'+str(ZIP))\n\nif __name__==\'__main__\':\n    main()\n', encoding='utf-8')
print('Pinned source verification and reconstructed inference scripts installed in /content/')


In [ ]:
import subprocess,sys
ret=subprocess.run([sys.executable,'/content/PACE_Phase11E5_Verified_Architecture_Forward.py'], check=False)
print('Runner exit code:',ret.returncode)
from pathlib import Path
import json,zipfile
z=Path('/content/PACE_Phase11E5_Results.zip')
if z.is_file():
    with zipfile.ZipFile(z) as fh:
        status=json.loads(fh.read('status.json'))
    print('PHASE11E5 STATUS:',status.get('status'))
    print('Positive canonical:',status.get('positive_canonical_count'))
    print('Unique positive canonical:',status.get('positive_unique_canonical_count'))
else:
    print('No result ZIP. Send the full traceback/output.')


In [ ]:
from google.colab import files
from pathlib import Path
p=Path('/content/PACE_Phase11E5_Results.zip')
if p.is_file():
    files.download(str(p))
else:
    print('Result ZIP missing; send printed traceback.')
